# 2.5. BGT Object Labeling

Labels all point objects in tiles that have been processed by **step 2 (Ground and Road fusion)**.

Sources used (in pipeline order):

| # | Object | Label | Source |
|---|--------|-------|--------|
| 1 | Car | 40 | BGT parkeervakken (polygons) |
| 2 | Bicycle rack | 88 | BGT fietsenrek |
| 3 | Bike parking | 88 | OSM fietsparkeren |
| 4 | Large container | 83 | BGT afval apart plaats |
| 5 | Bench | 80 | BGT bank |
| 6 | Rubbish bin | 81 | OOR afvalbakken |
| 7 | Street light | 60 | BGT lichtmast |
| 8 | Traffic light | 61 | BGT verkeersregelinstallatiepaal |
| 9 | Traffic sign | 62 | BGT verkeersbordpaal |
| 10 | Parking meter | 85 | OSM parkeermeters |
| 11 | Advertising sign | 89 | OSM reclameborden |
| 12 | Terrace | 91 | Amsterdam horeca WFS |
| 13 | Cable | 79 | CableFuser (geometric) |
| 14 | Tree | 30 | Bomen Atlas |

**Run this before step 3 (Extract 2D obstacles).**


## Config

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from config import LABELED_DIR, AHN_NPZ_DIR, BGT_DIR, BBOX_DIR, BOMEN_DIR, AFVAL_DIR, OSM_DIR, TERRAS_DIR, SETUP_TILECODES

tilecodes = SETUP_TILECODES

DIR_IN  = str(LABELED_DIR / "road_ground_labeled_")
DIR_OUT = str(LABELED_DIR / "bgt_labeled_")
AHN_DIR = AHN_NPZ_DIR

## Imports

In [ ]:
import utils.labels as LabelsModule
from utils.ahn_reader import PolygonNPZReader
from utils.bgt_readers import BGTPointReader
from utils.bgt_poly_reader import BGTPolyReader
from utils.pole_fuser import BGTPoleFuser
from utils.street_furniture_fuser import BGTStreetFurnitureFuser
from utils.car_fuser import CarFuser
from utils.cable_fuser import CableFuser
import utils.pipeline as Pipeline
import pandas as pd

Labels = LabelsModule.Labels


## CSVPointReader helper

In [ ]:
from utils.bgt_readers import get_bbox_from_polygon_file

class CSVPointReader:
    """
    Drop-in replacement for BGTPointReader for CSVs not in bgt_type/x/y format.
    filter_tile() returns [(x, y), ...] — same contract as BGTPointReader
    with return_types=False.
    """
    def __init__(self, csv_file, x_col='x', y_col='y', bbox_folder=None):
        self.df = pd.read_csv(csv_file)
        self.x_col = x_col
        self.y_col = y_col
        self.bbox_folder = Path(bbox_folder) if bbox_folder else None

    def _get_bbox(self, tilecode, padding):
        if self.bbox_folder is not None:
            return get_bbox_from_polygon_file(tilecode, self.bbox_folder, padding=padding)
        parts = tilecode.split('_')
        x_min = float(parts[0]) - padding
        y_min = float(parts[1]) - padding
        return ((x_min, y_min + 50 + 2*padding), (x_min + 50 + 2*padding, y_min))

    def filter_tile(self, tilecode, bgt_types=None, padding=0, return_types=False):
        ((bx_min, by_max), (bx_max, by_min)) = self._get_bbox(tilecode, padding)
        x, y = self.x_col, self.y_col
        df = self.df
        mask = (
            (df[x].astype(float) >= bx_min) & (df[x].astype(float) <= bx_max) &
            (df[y].astype(float) >= by_min) & (df[y].astype(float) <= by_max)
        )
        return [(float(row[x]), float(row[y])) for _, row in df[mask].iterrows()]


## Set up readers

In [ ]:
ahn_reader = PolygonNPZReader(AHN_DIR, caching=True)

poles_reader = BGTPointReader(
    bgt_file=str(BGT_DIR / "bgt_poles.csv"),
    bbox_folder=str(BBOX_DIR),
)

furniture_reader = BGTPointReader(
    bgt_file=str(BGT_DIR / "bgt_street_furniture.csv"),
    bbox_folder=str(BBOX_DIR),
)

parking_reader = BGTPolyReader(
    parkeervakken_file=str(BGT_DIR / "bgt_parkeervakken.json"),
    bbox_folder=str(BBOX_DIR),
)

bomen_reader = CSVPointReader(
    csv_file=str(BOMEN_DIR / "bomen_atlas.csv"),
    x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
)

oor_afval_reader = CSVPointReader(
    csv_file=str(AFVAL_DIR / "afvalbakken_oor.csv"),
    x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
)

# osm_fietsparkeren_reader = CSVPointReader(
#     csv_file=str(OSM_DIR / "osm_fietsparkeren.csv"),
#     x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
# )

# osm_parkeermeters_reader = CSVPointReader(
#     csv_file=str(OSM_DIR / "osm_parkeermeters.csv"),
#     x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
# )

# osm_reclameborden_reader = CSVPointReader(
#     csv_file=str(OSM_DIR / "osm_reclameborden.csv"),
#     x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
# )

terras_reader = CSVPointReader(
    csv_file=str(TERRAS_DIR / "terrassen_centroids.csv"),
    x_col='x', y_col='y', bbox_folder=str(BBOX_DIR),
)


## Set up fusers

In [ ]:
car_fuser = CarFuser(
    label=Labels.CAR,
    bgt_poly_reader=parking_reader,
    ahn_reader=ahn_reader,
    grid_size=0.1,
    min_component_size=100,
    overlap_perc=20,
    params={
        "min_height": 1.2, "max_height": 2.2,
        "min_width":  1.4, "max_width":  2.5,
        "min_length": 2.5, "max_length": 6.0,
    }
)

bike_rack_fuser = BGTStreetFurnitureFuser(
    label=Labels.BICYCLE_RACK,
    bgt_type="fietsenrek",
    bgt_reader=furniture_reader,
    ahn_reader=ahn_reader,
    max_dist=1.5,
    params={"min_height": 0.5, "max_height": 1.4,
            "min_width": 0.3, "max_width": 1.2,
            "min_length": 1.0, "max_length": 5.0},
)

# osm_bike_parking_fuser = BGTStreetFurnitureFuser(
#     label=Labels.BICYCLE_RACK,
#     bgt_type="osm_fietsparkeren",
#     bgt_reader=osm_fietsparkeren_reader,
#     ahn_reader=ahn_reader,
#     max_dist=1.5,
#     params={"min_height": 0.5, "max_height": 1.4,
#             "min_width": 0.3, "max_width": 1.2,
#             "min_length": 1.0, "max_length": 5.0},
# )

container_fuser = BGTStreetFurnitureFuser(
    label=Labels.LARGE_CONTAINER,
    bgt_type="afval apart plaats",
    bgt_reader=furniture_reader,
    ahn_reader=ahn_reader,
    max_dist=3.0,
    params={"min_height": 0.7, "max_height": 2.5,
            "min_width": 0.4, "max_width": 2.5,
            "min_length": 0.6, "max_length": 6.0},
)

bench_fuser = BGTStreetFurnitureFuser(
    label=Labels.CITY_BENCH,
    bgt_type="bank",
    bgt_reader=furniture_reader,
    ahn_reader=ahn_reader,
    max_dist=1.0,
    params={"min_height": 0.3, "max_height": 1.2,
            "min_width": 0.3, "max_width": 0.9,
            "min_length": 0.8, "max_length": 2.5},
)

bin_fuser = BGTStreetFurnitureFuser(
    label=Labels.RUBBISH_BIN,
    bgt_type="afvalbak",
    bgt_reader=oor_afval_reader,
    ahn_reader=ahn_reader,
    min_component_size=100,
    max_dist=1.0,
    params={"min_height": 0.4, "max_height": 1.2,
            "min_width": 0.2, "max_width": 0.8,
            "min_length": 0.2, "max_length": 0.8},
)

lamp_fuser = BGTPoleFuser(
    label=Labels.STREET_LIGHT,
    bgt_type="lichtmast",
    bgt_reader=poles_reader,
    ahn_reader=ahn_reader,
    params={"search_pad": 1.5, "max_dist": 1.2,
            "min_height": 2.0, "max_r": 0.3,
            "r_mult": 1.5, "label_height": 5.0},
)

traffic_light_fuser = BGTPoleFuser(
    label=Labels.TRAFFIC_LIGHT,
    bgt_type="verkeersregelinstallatiepaal",
    bgt_reader=poles_reader,
    ahn_reader=ahn_reader,
    params={"search_pad": 1.5, "max_dist": 1.2,
            "min_height": 2.0, "max_r": 0.3,
            "r_mult": 1.5, "label_height": 6.0},
)

traffic_sign_fuser = BGTPoleFuser(
    label=Labels.TRAFFIC_SIGN,
    bgt_type="verkeersbordpaal",
    bgt_reader=poles_reader,
    ahn_reader=ahn_reader,
    params={"search_pad": 1.5, "max_dist": 1.2,
            "min_height": 2.0, "max_r": 0.25,
            "r_mult": 1.5, "label_height": 4.0},
)

# parking_meter_fuser = BGTPoleFuser(
#     label=Labels.PARKING_METER,
#     bgt_type="osm_parkeermeter",
#     bgt_reader=osm_parkeermeters_reader,
#     ahn_reader=ahn_reader,
#     params={"search_pad": 1.0, "max_dist": 1.0,
#             "min_height": 0.5, "max_r": 0.2,
#             "r_mult": 1.5, "label_height": 1.5},
# )

# advertising_sign_fuser = BGTPoleFuser(
#     label=Labels.ADVERTISING_SIGN,
#     bgt_type="osm_reclamebord",
#     bgt_reader=osm_reclameborden_reader,
#     ahn_reader=ahn_reader,
#     params={"search_pad": 1.5, "max_dist": 1.2,
#             "min_height": 0.5, "max_r": 0.5,
#             "r_mult": 1.5, "label_height": 3.0},
# )

terrace_fuser = BGTStreetFurnitureFuser(
    label=Labels.TERRACE,
    bgt_type="terras",
    bgt_reader=terras_reader,
    ahn_reader=ahn_reader,
    max_dist=2.0,
    params={"min_height": 0.3, "max_height": 1.2,
            "min_width": 0.5, "max_width": 8.0,
            "min_length": 0.5, "max_length": 12.0},
)

cable_fuser = CableFuser(
    label=Labels.CABLE,
    cable_label=Labels.CABLE,
    tramcable_label=Labels.TRAM_CABLE,
    streetlight_label=Labels.ARMATUUR,
    ahn_reader=ahn_reader,
)

tree_fuser = BGTPoleFuser(
    label=Labels.TREE,
    bgt_type="boom",
    bgt_reader=bomen_reader,
    ahn_reader=ahn_reader,
    params={
        "search_pad": 2.0, "max_dist": 1.5,
        "min_height": 2.0, "max_r": 0.4,
        "r_mult": 1.5, "label_height": 8.0,
        "grow_crown": True,
        "crown_r": 5.0, "crown_height": 25.0,
        "crown_floor": 1.75,
        "crown_grid_size": 0.25, "crown_min_comp": 20,
        "crown_expand_step": 2.0, "crown_max_iter": 2,
    },
)


## Run labeling pipeline

In [ ]:
pipeline = Pipeline.Pipeline(
    processors=(
        car_fuser,              # cars first — stops crown growing absorbing nearby objects
        # bike_rack_fuser,        # BGT fietsenrek
        # osm_bike_parking_fuser, # OSM fietsparkeren
        container_fuser,
        bench_fuser,
        bin_fuser,
        lamp_fuser,
        # traffic_light_fuser,
        # traffic_sign_fuser,
        # parking_meter_fuser,
        # advertising_sign_fuser,
        terrace_fuser,
        cable_fuser,            # cables before trees — avoids crown absorption
        tree_fuser,             # tree crown growing runs last
    ),
    caching=False,
)

for tilecode in tilecodes:
    in_file  = f"{DIR_IN}{tilecode}.laz"
    out_file = f"{DIR_OUT}{tilecode}.laz"
    print(f"Processing {tilecode} ...")
    pipeline.process_file(in_file, out_file)
    print(f"  -> {out_file}")


## Visualise labeled objects

| Label | Meaning |
|-------|---------|
| 0 | Unknown | 1 | Road | 9 | Ground | 10 | Building |
| 30 | Tree | 40 | Car | 60 | Lamp post | 61 | Traffic light |
| 62 | Traffic sign | 79 | Cable | 80 | Bench | 81 | Bin |
| 83 | Container | 85 | Parking meter | 88 | Bike rack | 89 | Advertising sign |
| 91 | Terrace | | |


In [ ]:
import laspy
import numpy as np
import json

def load_tile(tilecode):
    pc = laspy.read(f"{DIR_OUT}{tilecode}.laz")
    x = np.asarray(pc.x, dtype=np.float32)
    y = np.asarray(pc.y, dtype=np.float32)
    if "label" in pc.point_format.extra_dimension_names:
        lbl = np.asarray(pc.label, dtype=np.int16)
    else:
        lbl = np.zeros(len(x), dtype=np.int16)
    return x, y, lbl

def load_tile_ring(tilecode):
    with open(BBOX_DIR / f"bbox_{tilecode}.geojson") as f:
        data = json.load(f)
    rings = []
    for feat in data["features"]:
        ring = feat["geometry"]["coordinates"][0]
        rings.append((list(zip(*ring))))
    return rings

LABEL_COLORS = {
     0: "#cccccc",  1: "#e15759",  9: "#c8a96e", 10: "#4e79a7",
    30: "#2ca02c", 40: "#d62728", 60: "#ff7f0e", 61: "#d62728",
    62: "#e377c2", 65: "#7f7f7f", 70: "#aec7e8", 79: "#1f77b4",
    80: "#6baed6", 81: "#9467bd", 83: "#8c564b", 85: "#f4c542",
    88: "#17becf", 89: "#ff4500", 90: "#ffbb78", 91: "#f4d03f",
}
LABEL_NAMES = {
     0: "Unknown",    1: "Road",        9: "Ground",      10: "Building",
    30: "Tree",      40: "Car",        60: "Lamp post",  61: "Traffic light",
    62: "Traffic sign", 70: "Tram cable", 79: "Cable",   80: "Bench",
    81: "Bin",       83: "Container",  85: "Parking meter",
    88: "Bike rack", 89: "Advert. sign", 90: "Armatuur", 91: "Terrace",
}


In [ ]:
try:
    import datashader as ds
    import datashader.transfer_functions as tf
    import pandas as pd
    HAS_DATASHADER = True
except ImportError:
    HAS_DATASHADER = False
    print("datashader not found — falling back to matplotlib")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

OBJECT_LABEL_IDS = [30, 40, 60, 61, 62, 70, 79, 80, 81, 83, 85, 88, 89, 91]

for tilecode in tilecodes:
    print(f"\n{tilecode}")
    x, y, lbl = load_tile(tilecode)
    rings = load_tile_ring(tilecode)

    unique, counts = np.unique(lbl, return_counts=True)
    for u, c in zip(unique, counts):
        name = LABEL_NAMES.get(int(u), str(u))
        print(f"  {name:<18} (label {int(u):>2}): {c:>10,}")

    fig, ax = plt.subplots(figsize=(10, 10))
    obj_mask = np.isin(lbl, OBJECT_LABEL_IDS)
    ax.scatter(x[~obj_mask], y[~obj_mask], c='#dddddd', s=0.2,
               linewidths=0, rasterized=True, zorder=1)
    for ring_xs, ring_ys in rings:
        ax.plot(ring_xs, ring_ys, 'k-', linewidth=1, zorder=2)
    handles = []
    for lbl_id in OBJECT_LABEL_IDS:
        m = lbl == lbl_id
        if not m.any(): continue
        color = LABEL_COLORS.get(lbl_id, '#aaaaaa')
        name  = LABEL_NAMES.get(lbl_id, str(lbl_id))
        ax.scatter(x[m], y[m], c=color, s=8, linewidths=0, zorder=3)
        handles.append(mpatches.Patch(color=color, label=f'{name} ({m.sum():,})'))
    ax.legend(handles=handles, loc='upper right', fontsize=8, framealpha=0.85)
    ax.set_aspect('equal')
    ax.set_title(f'{tilecode} — labeled objects')
    plt.tight_layout()
    plt.show()
